In [ ]:
import os
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

import re
from collections import Counter

import numpy as np
from datasets import load_dataset


def words(text):
    text = text.lower().replace("<br />", " ")
    return re.findall(r"[a-z']+", text)

train_rows = load_dataset("stanfordnlp/imdb", split="train", cache_dir="data/imdb")
test_rows = load_dataset("stanfordnlp/imdb", split="test", cache_dir="data/imdb")
train_rows = train_rows.shuffle(seed=42).select(range(2000))
test_rows = test_rows.shuffle(seed=42).select(range(500))

counts = Counter(w for row in train_rows for w in words(row["text"]))
vocab = {"<pad>": 0, "<unk>": 1}
for word, count in counts.most_common(7998):
    if count >= 2:
        vocab[word] = len(vocab)


def encode(rows, max_len=60):
    X = np.zeros((len(rows), max_len), dtype=np.int32)
    y = np.empty(len(rows), dtype=np.int32)

    for n, row in enumerate(rows):
        ids = [vocab.get(w, 1) for w in words(row["text"])[:max_len]]
        X[n, :len(ids)] = ids
        y[n] = row["label"]

    return X, y


X_train, y_train = encode(train_rows)
X_test, y_test = encode(test_rows)

In [ ]:
class SimpleLSTM:    def __init__(self, vocab_size, embed=32, hidden=32, seed=42):        rng = np.random.default_rng(seed)        self.H = hidden        d = embed + hidden        self.E  = (rng.standard_normal((vocab_size, embed)) * 0.1).astype("f")        self.W  = (rng.standard_normal((d, 4 * hidden)) / np.sqrt(d)).astype("f")        self.b  = np.zeros(4 * hidden, dtype="f")        self.b[:hidden] = 1                              # forget-bias        self.V  = (rng.standard_normal((hidden, 2)) / np.sqrt(hidden)).astype("f")        self.vb = np.zeros(2, dtype="f")    # ----- forward -----    def forward(self, X, train=False):        B, T = X.shape        H = self.H        h = np.zeros((B, H), dtype="f")        c = np.zeros_like(h)        cache = []        for t in range(T):            mask = (X[:, t] != 0)[:, None].astype("f")            z    = np.concatenate((self.E[X[:, t]], h * mask), axis=1)            f, i, g, o = np.split(z @ self.W + self.b, 4, axis=1)            f, i, o = sigmoid(f), sigmoid(i), sigmoid(o)            g = np.tanh(g)            c_new = f * c + i * g            h_new = o * np.tanh(c_new)            if train: cache.append((z, f, i, g, o, c, c_new, mask))            h = mask * h_new + (1 - mask) * h            c = mask * c_new + (1 - mask) * c        return h, (cache, h)        # h = last hidden == logits input    # ----- softmax cross-entropy + BPTT -----    def loss_and_grad(self, X, y):        (cache, h) = self.forward(X, train=True)[1]        B = len(y)        s = h @ self.V + self.vb        p = np.exp(s - s.max(1, keepdims=True))        p /= p.sum(1, keepdims=True)        loss = -np.log(p[np.arange(B), y] + 1e-12).mean()        dlog = p.copy()        dlog[np.arange(B), y] -= 1        dlog /= B        dE, dW, db = np.zeros_like(self.E), np.zeros_like(self.W), np.zeros_like(self.b)        dV, dvb   = h.T @ dlog, dlog.sum(0)        dh = dlog @ self.V.T        dc = np.zeros_like(dh)        d  = self.E.shape[1]        for t in range(X.shape[1] - 1, -1, -1):            z, f, i, g, o, c_prev, c_new, mask = cache[t]            dh  = dh * mask                              # padding 跳过本单元            dc += dh * o * (1 - np.tanh(c_new) ** 2)     # dL/dc            da = np.concatenate((                dc * c_prev * f * (1 - f),              # f-gate                dc * g      * i * (1 - i),              # i-gate                dc * i      * (1 - g * g),              # g-gate                dh * np.tanh(c_new) * o * (1 - o),      # o-gate            ), axis=1)            dW += z.T @ da            db += da.sum(0)            dz  = da @ self.W.T            np.add.at(dE, X[:, t], dz[:, :d] * mask)            dh = dz[:, d:]                               # -> 上一时刻            dc = dc * f                                  # -> 上一时刻 c        dE[0] = 0        return loss, dict(E=dE, W=dW, b=db, V=dV, vb=dvb)    # ----- sgd + global-norm clip -----    def update(self, grads, lr=0.05, clip=5.0):        norm = np.sqrt(sum(np.sum(g * g) for g in grads.values()))        scale = min(1.0, clip / (norm + 1e-8))        for name, g in grads.items():            getattr(self, name)[:] -= lr * scale * g    def predict(self, X):        return (self.forward(X)[0] @ self.V + self.vb).argmax(1)

In [ ]:
model = SimpleLSTM(len(vocab))

for epoch in range(3):
    order = np.random.permutation(len(X_train))
    total_loss = 0

    for start in range(0, len(order), 32):
        ids = order[start:start + 32]
        loss, grads = model.loss_and_grad(X_train[ids], y_train[ids])
        model.update(grads)
        total_loss += loss * len(ids)

    correct = 0
    for start in range(0, len(X_test), 64):
        pred = model.predict(X_test[start:start + 64])
        correct += np.sum(pred == y_test[start:start + 64])

    print(
        f"Epoch {epoch + 1}: "
        f"loss={total_loss / len(X_train):.4f}, "
        f"test_acc={correct / len(X_test):.4f}"
    )